# Parte 5 — Bonus: LoRA sobre BERTweet
### Workshop: Clasificación de Emociones en Twitter

BERTweet tiene ~110M parámetros — el doble que DistilBERT. LoRA permite adaptarlo con una fracción mínima de los parámetros.

BERTweet usa la arquitectura de BERT, cuyas proyecciones de atención tienen nombres distintos a DistilBERT:

| Modelo | Proyecciones |
|---|---|
| DistilBERT | `q_lin`, `k_lin`, `v_lin`, `out_lin` |
| BERTweet / BERT | `query`, `key`, `value`, `dense` |

**Prerequisito:** haber ejecutado `part-1-data.ipynb` y `part-2-pipeline.ipynb`

In [ ]:
# !pip install peft -q
%run part-2-pipeline.ipynb

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

MODEL_CHECKPOINT = "vinai/bertweet-base"
HF_REPO          = "your-username/tweeteval-emotion-bertweet-lora"  # <-- cambia esto
LR_LORA          = 2e-4   # más alto que full FT — pocos parámetros entrenables

## Dataset tokenizado

Reutilizamos el tokenizador de BERTweet del experimento anterior.

In [ ]:
from transformers import AutoTokenizer

tok_bertweet = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT, normalization=True)
ds_bertweet  = make_tokenized_dataset(tok_bertweet)
print(ds_bertweet)

## Inspección de módulos

Antes de aplicar LoRA, inspeccionamos los nombres de las capas lineales de BERTweet.

In [ ]:
from transformers import AutoModelForSequenceClassification

base_bt = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT, num_labels=NUM_LABELS,
    id2label=ID2LABEL, label2id=LABEL2ID)

print("Módulos lineales de BERTweet:")
for name, module in base_bt.named_modules():
    if isinstance(module, torch.nn.Linear):
        print(f"  {name}  [{module.in_features} → {module.out_features}]")

### 📝 TODO 5.1 — Configurar y aplicar LoRA a BERTweet

In [ ]:
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["query", "value"],
    modules_to_save=["classifier"],
)
model_bertweet_lora = get_peft_model(base_bt, lora_config)
model_bertweet_lora.print_trainable_parameters()


### 📝 TODO 5.2 — Entrenar BERTweet-LoRA

In [ ]:
trainer_lora = make_trainer(
    model_bertweet_lora, tok_bertweet, ds_bertweet,
    output_dir="./checkpoints/bertweet-lora", lr=LR_LORA,
)
trainer_lora.train()
plot_training_curves(trainer_lora, title="BERTweet-LoRA")
metrics_bertweet_lora = full_evaluation(trainer_lora, ds_bertweet["test"], model_name="BERTweet-LoRA")
metrics_bertweet_lora


import os
if os.environ.get("HF_TOKEN"):
    merged = model_bertweet_lora.merge_and_unload()
    merged.push_to_hub(HF_REPO, commit_message="BERTweet LoRA r=8 — merged")
    tok_bertweet.push_to_hub(HF_REPO)
    print(f"Modelo en https://huggingface.co/{HF_REPO}")
else:
    model_bertweet_lora.save_pretrained("./checkpoints/bertweet-lora/best")
    print("LoRA guardado localmente.")


### Resultados e interpretación — BERTweet + LoRA

| Métrica | Valor aprox. |
|---------|--------------|
| Parámetros entrenables | **888,580** (~0.65% del total) |
| Accuracy (test, 1 época) | **39.8%** |
| F1 macro (validación) | **~0.18** |

**Interpretación:** Con **solo 1 época** y LR alto para LoRA, el adaptador **no tuvo tiempo de converger** (accuracy peor que aleatorio estratificado en algunas clases). LoRA es eficiente en memoria y entrenamiento, pero necesita **más épocas** (p. ej. 3–5) o un LR afinado. En producción usaríamos el BERTweet fine-tune completo; LoRA sería útil para iterar rápido con más épocas o datos limitados de GPU.

In [ ]:
import os
if os.environ.get("HF_TOKEN"):
    merged = model_bertweet_lora.merge_and_unload()
    merged.push_to_hub(HF_REPO, commit_message="BERTweet LoRA r=8 — merged")
    tok_bertweet.push_to_hub(HF_REPO)
    print(f"Modelo en https://huggingface.co/{HF_REPO}")
else:
    model_bertweet_lora.save_pretrained("./checkpoints/bertweet-lora/best")
    print("LoRA guardado localmente.")
